In [ ]:
# Databricks notebook source
import dlt
from pyspark.sql import functions as F

# 1. Retrieve runtime parameters passed by the Databricks Job
bronze_table = spark.conf.get("pipeline.bronze_table")
silver_table_name = spark.conf.get("pipeline.silver_table_name")
preferred_loc_col = spark.conf.get("pipeline.preferred_loc_col", "location")
source_srid = int(spark.conf.get("pipeline.source_srid", "4326"))

silver_properties = {
    "delta.autoOptimize.optimizeWrite": "true",
    "delta.autoOptimize.autoCompact": "true",
    "delta.dataSkippingStatsColumns": "h3_r8"
}

# 2. Register only the target silver table
@dlt.table(
    name=silver_table_name,
    comment=f"Silver spatial table for {bronze_table}",
    table_properties=silver_properties
)
@dlt.expect_or_drop("valid_spatial_geom", "geom IS NULL OR st_isvalid(geom)")
@dlt.expect("h3_index_present", "geom IS NULL OR h3_r8 IS NOT NULL")
def process_single_silver():
    # Reads ONLY the specific newly ingested Bronze table
    df = dlt.read(bronze_table)
    
    # Unpack address and spatial fields
    df = (
        df.withColumn("street_address", F.get_json_object(F.get_json_object(F.col(preferred_loc_col), "$.human_address"), "$.address"))
          .withColumn("city", F.get_json_object(F.get_json_object(F.col(preferred_loc_col), "$.human_address"), "$.city"))
          .withColumn("state", F.get_json_object(F.get_json_object(F.col(preferred_loc_col), "$.human_address"), "$.state"))
          .withColumn("zip", F.get_json_object(F.get_json_object(F.col(preferred_loc_col), "$.human_address"), "$.zip"))
          .withColumn("__lat", F.expr(f"try_cast(get_json_object({preferred_loc_col}, '$.latitude') AS DOUBLE)"))
          .withColumn("__lon", F.expr(f"try_cast(get_json_object({preferred_loc_col}, '$.longitude') AS DOUBLE)"))
          .withColumn(
              "__g",
              F.expr(f"""
                  CASE 
                      WHEN __lon BETWEEN -180 AND 180 AND __lat BETWEEN -90 AND 90 AND NOT (__lon = 0 AND __lat = 0)
                      THEN st_point(__lon, __lat, {source_srid})
                      ELSE NULL 
                  END
              """)
          )
          .withColumn("geom", F.expr("__g" if source_srid == 4326 else f"st_transform(__g, 4326)"))
          .withColumn("geom_md", F.expr("__g" if source_srid == 6487 else f"st_transform(__g, 6487)"))
          .withColumn(
              "h3_r8",
              F.expr("""
                  CASE 
                      WHEN geom IS NULL OR st_isempty(geom) THEN NULL
                      WHEN abs(st_x(st_centroid(geom))) > 180 OR abs(st_y(st_centroid(geom))) > 90 THEN NULL
                      ELSE h3_longlatash3(st_x(st_centroid(geom)), st_y(st_centroid(geom)), 8)
                  END
              """)
          )
    )

    drop_cols = [c for c in df.columns if c in ["__g", "__lat", "__lon", preferred_loc_col]
                 or c.startswith("@computed_region") or c.startswith(":")]
    return df.drop(*drop_cols)